In [1]:
import pandas as pd

# TSV 로드
df = pd.read_csv(r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\rhapsody2_sav_db.tsv", sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

In [ ]:
import json
import os

# 입력 JSON
json_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\UniProtID_to_seq.json"
output_dir = r"E:\CAGI_data\blast_queries"
os.makedirs(output_dir, exist_ok=True)

# 불러오기
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

In [15]:
from Bio import SeqIO
import os

query_dir = r"E:\CAGI_data\fasta_files"
for fname in os.listdir(query_dir):
    if fname.endswith(".fasta"):
        path = os.path.join(query_dir, fname)
        records = list(SeqIO.parse(path, "fasta"))
        print(f"{fname}: {len(records)} sequences")
        for r in records:
            print(f"  > {r.id}, length = {len(r.seq)}")


A0A087WXS9.fasta: 1 sequences
  > A0A087WXS9, length = 549
A0A096LP55.fasta: 1 sequences
  > A0A096LP55, length = 91
A0A0A6YYG8.fasta: 1 sequences
  > A0A0A6YYG8, length = 113
A0A0A6YYL4.fasta: 1 sequences
  > A0A0A6YYL4, length = 1048
A0A0J9YWL9.fasta: 1 sequences
  > A0A0J9YWL9, length = 993
A0A0J9YY54.fasta: 1 sequences
  > A0A0J9YY54, length = 714
A0A0U1RQS6.fasta: 1 sequences
  > A0A0U1RQS6, length = 177
A0A1B0GWG4.fasta: 1 sequences
  > A0A1B0GWG4, length = 90
A0AUZ9.fasta: 1 sequences
  > A0AUZ9, length = 987
A0AV02.fasta: 1 sequences
  > A0AV02, length = 714
A0AV96.fasta: 1 sequences
  > A0AV96, length = 593
A0AVF1.fasta: 1 sequences
  > A0AVF1, length = 554
A0AVI2.fasta: 1 sequences
  > A0AVI2, length = 2057
A0AVI4.fasta: 1 sequences
  > A0AVI4, length = 362
A0AVK6.fasta: 1 sequences
  > A0AVK6, length = 867
A0AVT1.fasta: 1 sequences
  > A0AVT1, length = 1052
A0FGR9.fasta: 1 sequences
  > A0FGR9, length = 886
A0JNW5.fasta: 1 sequences
  > A0JNW5, length = 1464
A0MZ66.fasta: 1 

KeyboardInterrupt: 

In [ ]:
from Bio import SeqIO

# 2. FASTA에서 UniProt ID만 추출 (앞부분만 파싱)
fasta_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\uniprot_sprot.fasta"
available_ids = set()

for record in SeqIO.parse(fasta_path, "fasta"):
    try:
        entry_id = record.id.split("|")[1]
        available_ids.add(entry_id)
    except IndexError:
        continue

# 3. 존재 여부 확인
df["InSwissProt"] = df["UniProtID"].isin(available_ids)

# 4. 결과 요약
print(df["InSwissProt"].value_counts())


InSwissProt
True     117523
False         2
Name: count, dtype: int64


In [ ]:
import os
os.makedirs("msas", exist_ok=True)  # MSA 저장 폴더

In [5]:
import gzip
import shutil

src_path = r"E:\CAGI_data\uniprot_trembl.fasta.gz"
dst_path = r"E:\CAGI_data\uniprot_trembl.fasta"

with gzip.open(src_path, 'rb') as f_in:
    with open(dst_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

KeyboardInterrupt: 

안될거같음... 데이터가 너무 작음... BLAST는 계속 사용하는게 좋을듯...?
curl -o E:\CAGI_data\nr.gz ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/nr.gz

"C:\Users\Kunny\Desktop\정경건\tools\aria2\aria2c.exe" -x 16 -s 16 -c ^
ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/nr.gz ^
-d E:\CAGI_data ^
-o nr.gz


gz가 깨지는거 같아서 다시 curl -C - -o E:\CAGI_data\nr.gz ftp://ftp.ncbi.nlm.nih.gov/blast/db/FASTA/nr.gz로 받는중


In [8]:
import gzip
import shutil

input_path = r"E:\CAGI_data\nr_2.gz"
output_path = r"E:\CAGI_data\nr_2"

with gzip.open(input_path, 'rb') as f_in:
    with open(output_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Decompression complete:", output_path)

KeyboardInterrupt: 

In [ ]:
from Bio import SeqIO

# 2. FASTA에서 UniProt ID만 추출 (앞부분만 파싱)
fasta_path = r"C:\Users\Kunny\Research\Dataset\Missense Variant dataset\uniprot_sprot.fasta"
available_ids = set()

for record in SeqIO.parse(fasta_path, "fasta"):
    # FASTA header: >sp|A0A087WXS9|... 또는 >tr|... 이므로 split("|")[1]로 ID 추출
    try:
        entry_id = record.id.split("|")[1]
        available_ids.add(entry_id)
    except IndexError:
        continue

# 3. 존재 여부 확인
df["InSwissProt"] = df["UniProtID"].isin(available_ids)

# 4. 결과 요약
print(df["InSwissProt"].value_counts())


InSwissProt
True     117523
False         2
Name: count, dtype: int64


cd ~/BiConVarNet/

cat fasta_files/*.fasta > merged.fasta

grep -c "^>" merged.fasta

mmseqs createdb merged.fasta merged_db

#서버

mmseqs search merged_db uniref90_mmseqs merged_result_2 merged_tmp --threads 36 -e 0.001 --max-seqs 500

mmseqs result2msa merged_db uniref90_mmseqs merged_result_2 merged_msa_a3m \
  --msa-format-mode 5 \
  --threads 36 \
  --max-seq-id 0.95 \
  --qid 0.3 \
  --cov 0.3 \
  --filter-msa 1 \
  --filter-min-enable 100 \
  --diff 500


#PC

mmseqs search merged_db uniref90_mmseqs merged_result merged_tmp --threads 20 -e 0.001 --max-seqs 5000

mmseqs result2msa merged_db uniref90_mmseqs merged_result merged_msa_a3m \
  --msa-format-mode 5 \
  --threads 20 \
  --max-seq-id 0.95 \
  --qid 0.3 \
  --cov 0.3 \
  --filter-msa 1 \
  --filter-min-enable 100 \
  --diff 1000
